In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModelForSeq2SeqLM,GenerationConfig, AutoModelForCausalLM, DataCollatorWithPadding, DataCollatorForLanguageModeling
from trl import AutoModelForSeq2SeqLMWithValueHead, AutoModelForCausalLMWithValueHead, PPOTrainer, PPOConfig
#from trl.experimental.utils import create_reference_model
from datasets import Dataset, load_dataset
from peft import PeftModel, PeftConfig, LoraConfig, TaskType
import torch
import torchvision
import evaluate
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

In [2]:
model_name = "/scratch/general/vast/app-repo/huggingface/mistralai/Mistral-7B-Instruct-v0.2"
ppo_llm = AutoModelForCausalLMWithValueHead.from_pretrained(model_name) 
# ppo_llm = AutoModelForCausalLM.from_pretrained(model_name)   #"cuda:0")
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
df = pd.read_csv('best_movie_queries.csv')
dataset = Dataset.from_pandas(df)
# This dataset is very simple - just a bunch of variants of the question "What is the best movie?" These are prompts to the model
# and the response is analyzed later and penalized or rewarded.

In [4]:
def tokenize(sample):
    sample["input_ids"] = tokenizer.encode(sample["query"])
    return sample

# Tokenize prompts
dataset = dataset.map(tokenize, batched = False)
#We need to get rid of all other columns now.
dataset = dataset.remove_columns(['query'])
dataset.set_format(type = "torch")
# Split into training and testing datasets
# dataset = dataset.train_test_split(test_size=0.1)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [5]:
#torch.set_default_device('cpu')
# HuggingFace TRL PPO trainer configuration
config = PPOConfig(
    model_name = model_name, # not here in newer versions.
    learning_rate = 1.0e-6, # 1.41e-5,  #Smaller may be better - 1.0e-6, 
    mini_batch_size = 4,
    batch_size = 16
)

/uufs/chpc.utah.edu/common/home/u6040150/python_envs/llm_lora/lib/python3.12/site-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(


In [6]:
# Problem - we need a collator to fix unequal batches. But DataCollatorWithPadding crashes on strings.
collator = DataCollatorWithPadding(tokenizer=tokenizer)
# collator = DataCollatorForLanguageModeling(tokenizer=tokenizer)
#Unfortunately this library has seen a LOT of change over the last year. Commented out lines would be applicable to the new version.

# Inference parameters of the LLM generating responses
max_new_tokens = 500 
generation_kwargs = {
    "min_length": 5,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,   #pad_token_id,
    "max_new_tokens": max_new_tokens
}
# Inference parameters of the reward model
reward_kwargs = {
    "top_k": None,  
    "function_to_apply": "none", 
    "batch_size": 16}

# Set number of PPO iterations - This is really max # of batches - nrows / max_ppo_steps
max_ppo_steps = 20 

In [7]:
#Original test question:
test_question = tokenizer.encode('What is the best movie?')
test_question_tensor = torch.tensor([test_question])
default_answer = ppo_llm.generate(test_question_tensor, **generation_kwargs)
print(tokenizer.decode(default_answer, skip_special_tokens=True))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['What is the best movie? I think we can all agree that is an impossible question to answer definitively. It depends on personal taste, and different people hold the opinion that various films are the greatest ever made. With that said, I’d like to talk about ten movies that are widely regarded as favorites by critics, audiences and filmmakers.\n\n1. Citizen Kane (1941) – Orson Welles’ debut feature is considered a landmark achievement in cinema. Its complex narrative technique, innovative use of deep focus cinematography, and impressive performances from the cast make it a must-watch not just for cinemaphiles, but for anyone who loves a good story.\n\n2. The Godfather (1972) – Francis Ford Coppola’s epic crime drama is often cited as the best American film ever made. Marlon Brando’s iconic performance as the Godfather, the amazing supporting cast, the stunning cinematography, and the captivating storyline make it a film that is not easily forgotten.\n\n3. Rashomon (1950) – This Japane

In [7]:
# Next line crashes when run on CPU
ppo_trainer = PPOTrainer(config = config, # args = config,
                         model = ppo_llm,
                         ref_model = None, #ref_llm,
                         dataset = dataset, # train_dataset = dataset["train"],
                         #processing_class = tokenizer,
                         tokenizer=tokenizer,
                         #reward_model = reward_model.model,
                         #value_model = toxicity_model,
                         data_collator = collator
                        ) # where did this come from?
print(ppo_trainer.accelerator.device)

/uufs/chpc.utah.edu/common/home/u6040150/python_envs/llm_lora/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


cuda


In [8]:
# PPO loop
print("before start of step,batch", dataset)
epochs = 10
for epoch in tqdm(range(epochs), "epoch: "):
 for step, batch in tqdm(enumerate(ppo_trainer.dataloader)):
    #print(type(step),step) # <class 'int'> 0
    #print(type(batch), batch) # <class 'transformers.tokenization_utils_base.BatchEncoding'>
    # BatchEncoding appears to be a dictionary with input_ids and atteention_mask keys
    print("batch['input_ids'].shape", batch['input_ids'].shape)
    print("batch['attention_mask'].shape", batch['attention_mask'].shape) 
    # Stop after predefined number of steps
    if step >= max_ppo_steps:
        break

    # Produce a response for each prompt in the current batch 
    reward_tensors = []
    summary_tensors = [] 
    prompt_tensors = batch["input_ids"]
    print("before summary generation")
    for prompt_tensor in prompt_tensors:
        summary = ppo_trainer.generate(prompt_tensor, **generation_kwargs)
        # print('summary as returned', summary) #All the tokens
        summary_tensors.append(summary.squeeze()[-max_new_tokens:])
        #print('originaly summary_tensors.append(',summary.squeeze()[-max_new_tokens:]) #Still tokens
        decoded_response = tokenizer.decode(summary, skip_special_tokens=True)
        reward = 1
        # I am tired of everyone saying Citizen Kane and The Godfather are the best movies ever made.
        # I will penalize any model saying so.
        if 'Citizen Kane' in decoded_response or 'The Godfather' in decoded_response:
            reward = -1
        reward_tensors.append(torch.tensor(float(reward)))
    # first param is supposed to be a list so we have to convert the 2D tensor.
    stats = ppo_trainer.step(list(prompt_tensors), summary_tensors, reward_tensors)
    ppo_trainer.log_stats(stats, batch, reward_tensors)

    # Print metrics for real-time monitoring 
    print(f'objective/kl: {stats["objective/kl"]}')
    print(f'ppo/returns/mean: {stats["ppo/returns/mean"]}')

before start of step,batch Dataset({
    features: ['input_ids'],
    num_rows: 50
})


epoch:   0%|          | 0/10 [00:00<?, ?it/s]
0it [00:00, ?it/s][transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


batch['input_ids'].shape torch.Size([16, 13])
batch['attention_mask'].shape torch.Size([16, 13])
before summary generation


/uufs/chpc.utah.edu/common/home/u6040150/python_envs/llm_lora/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1392: UserWarning: The game logs will not be logged because the batch does not contain the keys 'query' and 'response'. 
  warnings.warn(

1it [02:27, 147.71s/it]

objective/kl: 0.0
ppo/returns/mean: -0.056345097720623016
batch['input_ids'].shape torch.Size([16, 12])
batch['attention_mask'].shape torch.Size([16, 12])
before summary generation


...

objective/kl: 24.67871856689453
ppo/returns/mean: -0.7466588020324707
batch['input_ids'].shape torch.Size([16, 14])
batch['attention_mask'].shape torch.Size([16, 14])
before summary generation



2it [03:25, 103.81s/it]

objective/kl: 51.442138671875
ppo/returns/mean: -1.313729166984558
batch['input_ids'].shape torch.Size([16, 13])
batch['attention_mask'].shape torch.Size([16, 13])
before summary generation



3it [05:15, 105.22s/it]
epoch: 100%|██████████| 10/10 [54:43<00:00, 328.34s/it]

objective/kl: 27.745162963867188
ppo/returns/mean: -1.059664249420166


In [9]:
test_question = tokenizer.encode('What is the best movie?')
test_question_tensor = [{"input_ids": torch.tensor(test_question)}]
print(type(test_question_tensor), test_question_tensor)
batch = collator(test_question_tensor)
print(batch)
default_answer = ppo_trainer.generate(batch['input_ids'][0].to('cuda'), **generation_kwargs)
print(tokenizer.decode(default_answer, skip_special_tokens=True))
#Sadly I only got rid of The Godfather as Citizen Kane is still mentioned.

<class 'list'> [{'input_ids': tensor([    1,  1824,   349,   272,  1489,  5994, 28804])}]
{'input_ids': tensor([[    1,  1824,   349,   272,  1489,  5994, 28804]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}
['What is the best movie? It’s a trick question because everyone has different preferences when it comes to movies. Some people might say Citizen Kane is the best movie ever made because of its innovative storytelling techniques. Others might argue that The Shawshank Redemption is the best movie because of its emotional depth and powerful performances. Still, others might argue that Star Wars is the best movie because of its epic storytelling and iconic characters. Ultimately, the best movie is subjective and depends on personal tastes and preferences.']


In [ ]:
# Compute aggregate toxicity score (mean, std dev) of the original model on the test set
mean_before, std_before = evaluate_toxicity(model=ref_llm,
                                            toxicity_evaluator=toxicity_evaluator,
                                            tokenizer=tokenizer,
                                            dataset=dataset["test"],
                                            num_samples=10)

# Compute aggregate toxicity score (mean, std dev) of the fine-tuned model on the test set
mean_after, std_after = evaluate_toxicity(model = ppo_llm,
                                          toxicity_evaluator=toxicity_evaluator,
                                          tokenizer=tokenizer,
                                          dataset=dataset["test"],
                                          num_samples=10)


In [10]:
ppo_trainer.save_pretrained("/scratch/general/vast/u6040150/best_movie_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/uufs/chpc.utah.edu/common/home/u6040150/python_envs/llm_lora/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1431: UserWarning: Cannot retrieve user information assuming you are running in offline mode.
  warnings.warn("Cannot retrieve user information assuming you are running in offline mode.")


In [ ]:
# Load Helpfulness/Harmfulness dataset from Anthropic
# dataset_name = "Anthropic/hh-rlhf"
# Engineer the prompt and build the training/test dataset
# dataset = load_dataset(dataset_name, split="train")
# dataset = dataset.remove_columns("rejected")
# dataset = dataset.rename_column("chosen", "dialogue")
# dataset = dataset.filter(lambda x: len(x["dialogue"]) > 100 and
#                          len(x["dialogue"]) <= 500, batched=False) # Limit size of dialogues
# I glossed over this in datasets and maybe shouldn't have. AI sez this about datasets
# TypeError: If you provide a function designed for batches (returning a list) while batched=False, 
# the filter will fail because it expects a single boolean for every call.
# Reverse would be is that list is expected?

#use_cpu = True) # does not work anyway
# Some param explanations: n Hugging Face PPO configurations, num_train_epochs dictates the total passes over the entire dataset, 
# while num_ppo_epochs determines the number of optimization passes per PPO batch. Essentially, num_train_epochs controls the 
# overall training lifecycle, whereas num_ppo_epochs is a micro-optimization step within reinforcement learning.

# Instantiate the PPO trainer
# PPOTrainer requirements:
# model (torch.nn.Module) — Model to be trained. This is the policy model.
# processing_class (PreTrainedTokenizerBase, BaseImageProcessor, FeatureExtractionMixin or ProcessorMixin) — Class to process the data.
# reward_model (torch.nn.Module) — Reward model used to compute the rewards.
#  value_model (torch.nn.Module) — Value model used to predict the value of a state.
#collator = DataCollatorWithPadding(tokenizer=tokenizer)
#dataset = dataset.remove_columns(['chosen', 'query'])
# So "problem" is that ppo_trainer.dataloader has strings in dataset. Apparently at this point they're supposed to
# be tokenized?!? The problem apparently is that sentiment has strings in it. I can probably just drop that col.
# BUT I DID ALREADY!!!! Oh is it the fact that "chosen" is still in it?!?

# BELOW is original summary loop - PROBLEM is that model.policy.generate wants a block now, not a single row
    #for prompt_tensor in prompt_tensors:
    #    print(prompt_tensor.shape, prompt_tensor)
    #    summary = ppo_trainer.model.policy.generate(prompt_tensor, **generation_kwargs)
    #    summary_tensors.append(summary.squeeze()[-max_new_tokens:])

    #For newer versions of trl
    # summary = ppo_trainer.model.policy.generate(prompt_tensors, **generation_kwargs)
    # We still need to do this per item to go along with original code below.
    # for s in summary:
    #  summary_tensors.append(s.squeeze()[-max_new_tokens:])    

#ppo_trainer.train()